---
title: CONUS 404 diagnostic plots
author: Harsha R. Hampapura
date: 09-02-2026
---

### Data Access

- This notebook illustrates how to make diagnostic plots using the CONUS 404 dataset hosted on NCAR's Geoscience Data Exchange (GDEX).
- https://gdex.ucar.edu/datasets/d559000/
- This data is open access and can be accessed via 3 protocols
  1) POSIX (if you have access to NCAR's HPC systems: Casper or Derecho)
  2) HTTPS
  3) OSDF using intake-ESM catalogs.
- Learn about intake-ESM catalogs: https://intake-esm.readthedocs.io/en/stable/ 

In [2]:
# Imports 
import intake
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [3]:
# Catalog URLs
cat_url     = 'https://data.gdex.ucar.edu/d559000/catalogs/d559000-posix.json' # POSIX access
# cat_url     = 'https://osdf-director.osg-htc.org/ncar/gdex/d559000/catalogs/d559000-osdf.json'
print(cat_url)

https://data.gdex.ucar.edu/d559000/catalogs/d559000-posix.json


## Section 2: Select Dask Cluster

#### Select the Dask cluster type
The default will be LocalCluster as that can run on any system.

If running on a HPC computer with a PBS Scheduler, set to True. Otherwise, set to False.

In [ ]:
USE_PBS_SCHEDULER = False

If running on Jupyter server with Dask Gateway configured, set to True. Otherwise, set to False.

In [5]:
USE_DASK_GATEWAY = False

#### Python function for a PBS cluster

In [ ]:
# Create a PBS cluster object
def get_pbs_cluster():
    """ Create cluster through dask_jobqueue.   
    """
    # Set up your scratch folder path
    username       = os.environ["USER"]
    glade_scratch  = "/glade/derecho/scratch/" + username
    print(glade_scratch)
    from dask_jobqueue import PBSCluster
    cluster = PBSCluster(
        job_name = 'dask-osdf-24',
        cores = 1,
        memory = '4GiB',
        processes = 1,
        local_directory = lustre_scratch + '/dask/spill',
        log_directory = lustre_scratch + '/dask/logs/',
        resource_spec = 'select=1:ncpus=1:mem=4GB',
        queue = 'casper',
        walltime = '3:00:00',
        #interface = 'ib0'
        interface = 'ext'
    )
    return cluster

#### Python function for a Gateway Cluster

In [7]:
def get_gateway_cluster():
    """ Create cluster through dask_gateway
    """
    from dask_gateway import Gateway

    gateway = Gateway()
    cluster = gateway.new_cluster()
    cluster.adapt(minimum=2, maximum=4)
    return cluster

In [8]:
def get_local_cluster():
    """ Create cluster using the Jupyter server's resources
    """
    from distributed import LocalCluster, performance_report
    cluster = LocalCluster()    

    cluster.scale(6)
    return cluster

#### Python logic for a Local Cluster
This uses True/False boolean logic based on the variables set in the previous cells

In [9]:
# Obtain dask cluster in one of three ways
if USE_PBS_SCHEDULER:
    cluster = get_pbs_cluster()
elif USE_DASK_GATEWAY:
    cluster = get_gateway_cluster()
else:
    cluster = get_local_cluster()

# Connect to cluster
from distributed import Client
client = Client(cluster)

In [10]:
# Scale the cluster and display cluster dashboard URL
n_workers =8
cluster.scale(n_workers)
client.wait_for_workers(n_workers = n_workers)
cluster

PBSCluster(bfdd887b, 'tcp://128.117.208.98:41657', workers=8, threads=8, memory=32.00 GiB)

## Load CONUS 404 data from GDEX using an intake catalog

In [7]:
col = intake.open_esm_datastore(cat_url)
col

,unique
path,79
variable,206
format,1
short_name,206
long_name,119
units,33
start_time,41
end_time,41
level,1
level_units,1


- col.df turns the catalog object into a pandas dataframe!
- (Actually, it accesses the dataframe attribute of the catalog)

In [8]:
col.df

,path,variable,format,short_name,long_name,units,start_time,end_time,level,level_units,frequency
0,/glade/campaign/collections/rda/data/d559000/k...,ACDEWC,reference,ACDEWC,"Accumulated canopy dew rate, accumulated over ...",mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
1,/glade/campaign/collections/rda/data/d559000/k...,ACDRIPR,reference,ACDRIPR,"Accumulated canopy precipitation drip rate, ac...",mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
2,/glade/campaign/collections/rda/data/d559000/k...,ACDRIPS,reference,ACDRIPS,"Accumulated canopy snow drip rate, accumulated...",mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
3,/glade/campaign/collections/rda/data/d559000/k...,ACECAN,reference,ACECAN,Accumulated net evaporation of canopy water (e...,mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
4,/glade/campaign/collections/rda/data/d559000/k...,ACEDIR,reference,ACEDIR,Accumulated net soil evaporation or snowpack s...,mm,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
...,...,...,...,...,...,...,...,...,...,...,...
8574,/glade/campaign/collections/rda/data/d559000/k...,V,reference,V,<NA>,m s-1,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
8575,/glade/campaign/collections/rda/data/d559000/k...,W,reference,W,<NA>,m s-1,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
8576,/glade/campaign/collections/rda/data/d559000/k...,Z,reference,Z,<NA>,m2 s-2,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
8577,/glade/campaign/collections/rda/data/d559000/k...,ilev,reference,ilev,vertical stagger levels,Dimensionless,2020-10-01,2021-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00


## Select data and plot

#### What if you don't know the variable names ?
- Use pandas logic to print out the short_name and long_name

In [9]:
col.df[['variable','long_name']]

,variable,long_name
0,ACDEWC,"Accumulated canopy dew rate, accumulated over ..."
1,ACDRIPR,"Accumulated canopy precipitation drip rate, ac..."
2,ACDRIPS,"Accumulated canopy snow drip rate, accumulated..."
3,ACECAN,Accumulated net evaporation of canopy water (e...
4,ACEDIR,Accumulated net soil evaporation or snowpack s...
...,...,...
8574,V,<NA>
8575,W,<NA>
8576,Z,<NA>
8577,ilev,vertical stagger levels


- We notice that long_name is not available for some variables like 'V'
- In such cases, please look at the wrfout_datadictionary file on this page https://gdex.ucar.edu/datasets/d559000/documentation/#

### Temperature
- Plot temperature for a random date

In [10]:
cat_temp = col.search(variable='T2')
cat_temp.df.head()

,path,variable,format,short_name,long_name,units,start_time,end_time,level,level_units,frequency
0,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1979-10-01,1980-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
1,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1980-10-01,1981-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
2,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1981-10-01,1982-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
3,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1982-10-01,1983-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
4,/glade/campaign/collections/rda/data/d559000/k...,T2,reference,T2,<NA>,K,1983-10-01,1984-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00


In [11]:
cat_temp.df.head().values

array([['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1980.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1979-10-01',
        '1980-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1981.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1980-10-01',
        '1981-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1982.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1981-10-01',
        '1982-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1983.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1982-10-01',
        '1983-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:00'],
       ['/glade/campaign/collections/rda/data/d559000/kerchunk/wy1984.2d.json',
        'T2', 'reference', 'T2', <NA>, 'K', '1983-10-01',
        '1984-09-30 23:00:00', <NA>, <NA>, '0 days 01:00:0

In [12]:
# %%time
# test = xr.open_dataset('/gdex/data/d559000/kerchunk/wy1980.2d.json')
# test

In [ ]:
ds = xr.open_dataset('https://data.gdex.ucar.edu/d559000/kerchunk/wy1980.2d-osdf.json', engine='kerchunk')

/glade/u/home/harshah/.conda/envs/osdf/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/glade/u/home/harshah/.conda/envs/osdf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
date = "1980-09-30"
test.T2.sel(Time=date,method='nearest').values

- The data is organized in (virtual) zarr stores with one water year's worth of data in one file
- Select a year. This is done by selcting the start time to be Oct 1 of that year or the end time to be Sep 30 of the same year
- This also means that if you want to request data for other days, say Jan 1 for the year YYYY, you first have to load the data for one year i.e., YYYY and then select the data for that particular day. This example is discussed below.


In [ ]:
date = "2020-10-01"
# year = "2021"
cat_temp_subset = cat_temp.search(start_time = date)
cat_temp_subset

### Load data into xarray

In [ ]:
%%time
# Load catalog entries for subset into a dictionary of xarray datasets, and open the first one.
dsets = cat_temp_subset.to_dataset_dict(xarray_open_kwargs={'engine':'kerchunk',"chunks": {}})
#
print(f"\nDataset dictionary keys:\n {dsets.keys()}")

In [ ]:
# Load the first dataset and display a summary.
dataset_key = list(dsets.keys())[0]
# store_name = dataset_key + ".zarr"
print(dsets.keys())
ds = dsets[dataset_key]
ds = ds.T2
ds

In [ ]:
%%time
desired_time = "2021-01-01T00"
ds.sel(Time=desired_time,method='nearest').plot(cmap='inferno')

In [ ]:
cluster.close()